# 01: Load the raw Favorita files into Databricks

The six raw Kaggle files go straight from Kaggle into Databricks. Nothing is uploaded from a laptop.

This notebook builds the **bronze** layer on Databricks: the same six raw tables as the PostgreSQL `raw` schema.

**Steps:**
1. Create a volume for the raw files.
2. Download the six files from Kaggle.
3. Unpack the `.7z` archives.
4. Create one table per file.
5. Check the row counts against the PostgreSQL load.

## 1. Storage

A **volume** is a storage folder for files inside Databricks. The volume `landing` holds raw files, with one folder per data source.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.raw;
CREATE VOLUME IF NOT EXISTS workspace.raw.landing;

## 2. Download from Kaggle

The Kaggle command-line tool downloads the files. It needs a Kaggle API token.

**Never commit a real token.** Paste it only while the cell runs, then delete it.

Only six files are downloaded. `test.csv` and `sample_submission.csv` exist only for the Kaggle contest.

In [ ]:
%pip install --upgrade kaggle py7zr

In [ ]:
%restart_python

In [ ]:
import os
import subprocess

# Kaggle API token: paste it here only while running, then delete it before saving or exporting.
# Never commit a real token. In a team workspace, it would come from a Databricks secret scope.
os.environ["KAGGLE_API_TOKEN"] = "PASTE_YOUR_KAGGLE_TOKEN_HERE"

# Landing zone for raw files, one folder per data source
download_folder = "/Volumes/workspace/raw/landing/favorita"
os.makedirs(download_folder, exist_ok=True)

# The 6 files Shelfcast uses (test and sample_submission are for the Kaggle contest only)
files = [
    "stores.csv.7z",
    "items.csv.7z",
    "oil.csv.7z",
    "holidays_events.csv.7z",
    "transactions.csv.7z",
    "train.csv.7z",
]

for file_name in files:
    result = subprocess.run(
        ["kaggle", "competitions", "download",
         "-c", "favorita-grocery-sales-forecasting",
         "-f", file_name,
         "-p", download_folder],
        capture_output=True,
        text=True,
    )
    print(file_name, "| return code:", result.returncode)
    if result.returncode != 0:
        print(result.stderr)

print(os.listdir(download_folder))

stores.csv.7z | return code: 0
items.csv.7z | return code: 0
oil.csv.7z | return code: 0
holidays_events.csv.7z | return code: 0
transactions.csv.7z | return code: 0
train.csv.7z | return code: 0
['stores.csv.7z', 'items.csv.7z', 'oil.csv.7z', 'holidays_events.csv.7z', 'transactions.csv.7z', 'train.csv.7z']


## 3. Unpack the archives

Kaggle stores each file as a `.7z` archive. `py7zr` unpacks them into the same folder.

In [ ]:
import os
import py7zr

download_folder = "/Volumes/workspace/raw/landing/favorita"

# Extract every .7z file in the folder
for file_name in os.listdir(download_folder):
    if file_name.endswith(".7z"):
        archive_path = download_folder + "/" + file_name
        with py7zr.SevenZipFile(archive_path, mode="r") as archive:
            archive.extractall(path=download_folder)
        print("Extracted", file_name)

# Show each CSV and its size in MB
for file_name in os.listdir(download_folder):
    if file_name.endswith(".csv"):
        size_mb = os.path.getsize(download_folder + "/" + file_name) / 1024 / 1024
        print(file_name, "|", round(size_mb, 1), "MB")

Extracted stores.csv.7z
Extracted items.csv.7z
Extracted oil.csv.7z
Extracted holidays_events.csv.7z
Extracted transactions.csv.7z
Extracted train.csv.7z
stores.csv | 0.0 MB
items.csv | 0.1 MB
oil.csv | 0.0 MB
holidays_events.csv | 0.0 MB
transactions.csv | 1.5 MB
train.csv | 4765.9 MB


## 4. Create the raw tables

`read_files` reads a CSV file into a table and guesses the column types.

`train` gets an explicit schema instead. The types match the PostgreSQL `raw.train` table: `FLOAT` here is a 4-byte number, the same as `real` in PostgreSQL.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.raw.stores AS
SELECT * FROM read_files('/Volumes/workspace/raw/landing/favorita/stores.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.raw.items AS
SELECT * FROM read_files('/Volumes/workspace/raw/landing/favorita/items.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.raw.oil AS
SELECT * FROM read_files('/Volumes/workspace/raw/landing/favorita/oil.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.raw.holidays_events AS
SELECT * FROM read_files('/Volumes/workspace/raw/landing/favorita/holidays_events.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.raw.transactions AS
SELECT * FROM read_files('/Volumes/workspace/raw/landing/favorita/transactions.csv', format => 'csv', header => true);

num_affected_rows,num_inserted_rows


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.raw.train AS
SELECT * FROM read_files(
  '/Volumes/workspace/raw/landing/favorita/train.csv',
  format => 'csv',
  header => true,
  schema => 'id INT, date DATE, store_nbr SMALLINT, item_nbr INT, unit_sales FLOAT, onpromotion BOOLEAN'
);

num_affected_rows,num_inserted_rows


## 5. Check against the PostgreSQL load

The expected numbers come from the profiling notebook and the PostgreSQL load checks.

In [ ]:
%sql
-- Expected (basis: the profiling notebook and the PostgreSQL load checks):
-- stores 54, items 4,100, oil 1,218, holidays_events 350, transactions 83,488, train 125,497,040
SELECT 'stores' AS table_name, COUNT(*) AS n_rows FROM workspace.raw.stores
UNION ALL SELECT 'items', COUNT(*) FROM workspace.raw.items
UNION ALL SELECT 'oil', COUNT(*) FROM workspace.raw.oil
UNION ALL SELECT 'holidays_events', COUNT(*) FROM workspace.raw.holidays_events
UNION ALL SELECT 'transactions', COUNT(*) FROM workspace.raw.transactions
UNION ALL SELECT 'train', COUNT(*) FROM workspace.raw.train;

table_name,n_rows
stores,54
items,4100
oil,1218
holidays_events,350
transactions,83488
train,125497040


In [ ]:
%sql
-- Expected (basis: the PostgreSQL load check):
-- false 96,028,767 / true 7,810,622 / NULL 21,657,651
SELECT onpromotion, COUNT(*) AS n_rows
FROM workspace.raw.train
GROUP BY onpromotion
ORDER BY onpromotion;

onpromotion,n_rows
null,21657651
false,96028767
true,7810622


### Findings

- **Row counts:** all six tables match the PostgreSQL load exactly. `train` has 125,497,040 rows.
- **Promotion flag:** the `onpromotion` split matches exactly: 96,028,767 false, 7,810,622 true, 21,657,651 NULL.